# External-package interfaces

Frozen-core BH with a 1.232 Å bond length in STO-3G and the Jordan–Wigner mapping. The B 1s spatial orbital is frozen in each interface. Run with the Python environment used for VarSI.

Shared configuration and helper functions using the fast VarSI implementation. Each interface explicitly selects the covariance method used to build its groupings; all reported results use exact covariances.

In [ ]:
import os
import time
from itertools import combinations
from types import SimpleNamespace
import numpy as np
import pandas as pd
import tequila as tq

from openfermion.linalg import get_ground_state, get_sparse_operator
from openfermion.utils import count_qubits
from tequila.hamiltonian import QubitHamiltonian

import fast_VarSI as varsi

CONDITION = "fc"
MAX_SWEEPS = 100
try:
    AVAILABLE_CPUS = len(os.sched_getaffinity(0))
except (AttributeError, OSError):
    AVAILABLE_CPUS = os.cpu_count() or 1
COVARIANCE_WORKERS = min(12, AVAILABLE_CPUS)
FAST_OPTIONS = SimpleNamespace(
    max_sweeps=MAX_SWEEPS,
    allow_new_groups=False,
    ordered_consider_new_groups=False,
)

BH_SYMBOLS = ["B", "H"]
BH_COORDINATES = np.array(
    [[0.0, 0.0, -0.616], [0.0, 0.0, 0.616]], dtype=float
)
BH_GEOMETRY = list(zip(BH_SYMBOLS, map(tuple, BH_COORDINATES)))
BH_ACTIVE_ELECTRONS = 4
BH_FROZEN_ORBITALS = [0]
BH_ACTIVE_ORBITALS = [1, 2, 3, 4, 5]


def spin_label(mode, n_modes, spin_ordering):
    if spin_ordering == "interleaved":
        return mode % 2
    if spin_ordering == "blocked":
        return 0 if mode < n_modes // 2 else 1
    raise ValueError("spin_ordering must be 'interleaved' or 'blocked'.")


def closed_shell_reference(n_modes, n_electrons, spin_ordering="interleaved"):
    if n_modes % 2 or n_electrons % 2 or not 0 <= n_electrons <= n_modes:
        raise ValueError(
            "CISD requires an even number of modes and electrons."
        )
    occupations = [0] * n_modes
    occupied_spatial = n_electrons // 2
    if spin_ordering == "interleaved":
        for orbital in range(occupied_spatial):
            occupations[2 * orbital] = occupations[2 * orbital + 1] = 1
    elif spin_ordering == "blocked":
        n_spatial = n_modes // 2
        for orbital in range(occupied_spatial):
            occupations[orbital] = occupations[n_spatial + orbital] = 1
    else:
        raise ValueError("spin_ordering must be 'interleaved' or 'blocked'.")
    return tuple(occupations)


def generate_cisd_determinants(
    n_modes, n_electrons, spin_ordering="interleaved"
):
    reference = closed_shell_reference(n_modes, n_electrons, spin_ordering)
    occupied = [mode for mode, value in enumerate(reference) if value]
    virtual = [mode for mode, value in enumerate(reference) if not value]
    determinants = {reference}
    for excitation_rank in (1, 2):
        for removed in combinations(occupied, excitation_rank):
            for added in combinations(virtual, excitation_rank):
                removed_spins = sorted(
                    spin_label(mode, n_modes, spin_ordering)
                    for mode in removed
                )
                added_spins = sorted(
                    spin_label(mode, n_modes, spin_ordering)
                    for mode in added
                )
                if removed_spins != added_spins:
                    continue
                determinant = list(reference)
                for mode in removed:
                    determinant[mode] = 0
                for mode in added:
                    determinant[mode] = 1
                determinants.add(tuple(determinant))
    return sorted(determinants)


def occupations_to_basis_index(occupations):
    index = 0
    n_qubits = len(occupations)
    for qubit, occupied in enumerate(occupations):
        if occupied:
            index |= 1 << (n_qubits - 1 - qubit)
    return index


def cisd_statevector(
    qubit_operator,
    n_qubits,
    n_electrons,
    spin_ordering="interleaved",
    validation_tol=1.0e-8,
):
    determinants = generate_cisd_determinants(
        n_qubits, n_electrons, spin_ordering
    )
    basis_indices = np.asarray(
        [
            occupations_to_basis_index(determinant)
            for determinant in determinants
        ],
        dtype=int,
    )
    sparse_hamiltonian = get_sparse_operator(
        qubit_operator, n_qubits=n_qubits
    )
    subspace_hamiltonian = sparse_hamiltonian[basis_indices, :][
        :, basis_indices
    ].toarray()
    subspace_hamiltonian = 0.5 * (
        subspace_hamiltonian + subspace_hamiltonian.conjugate().T
    )
    eigenvalues, eigenvectors = np.linalg.eigh(subspace_hamiltonian)
    ground_index = int(np.argmin(eigenvalues))
    energy = float(np.real(eigenvalues[ground_index]))

    state = np.zeros(2**n_qubits, dtype=complex)
    state[basis_indices] = eigenvectors[:, ground_index]
    state /= np.linalg.norm(state)
    expectation = np.vdot(state, sparse_hamiltonian.dot(state))
    if abs(expectation - energy) > validation_tol:
        raise RuntimeError(
            "CISD subspace energy did not match the state expectation."
        )
    metadata = {
        "basis_size": len(determinants),
        "expectation": float(np.real(expectation)),
    }
    return energy, state, metadata


def covariance_dictionary_from_state(terms, state, n_qubits, label):
    state = np.asarray(state, dtype=complex).reshape(-1)
    state /= np.linalg.norm(state)
    start = time.perf_counter()
    covariances, single_expectations = varsi.build_covariance_dictionary(
        terms,
        state,
        n_qubits,
        max_workers=COVARIANCE_WORKERS,
        chunksize=128,
    )
    runtime = time.perf_counter() - start
    print(
        f"{label} covariance dictionary: {len(covariances)} entries; "
        f"fast builder with {COVARIANCE_WORKERS} workers; "
        f"runtime={runtime:.3f} s"
    )
    return covariances, single_expectations


def build_covariance_problem(qubit_operator, method):
    method = str(method).strip().upper()
    if method == "FCI":
        method = "EXACT"
    if method not in {"EXACT", "CISD"}:
        raise ValueError("method must be 'exact' or 'CISD'.")

    n_qubits = int(count_qubits(qubit_operator))
    sparse_hamiltonian = get_sparse_operator(qubit_operator, n_qubits=n_qubits)
    exact_energy, exact_state = get_ground_state(sparse_hamiltonian)
    tequila_hamiltonian = QubitHamiltonian.from_openfermion(qubit_operator)
    all_terms = varsi.make_terms(qubit_operator, n_qubits)
    terms = [term for term in all_terms if term.pauli_tuple]

    if method == "CISD":
        grouping_energy, grouping_state, cisd_metadata = cisd_statevector(
            qubit_operator,
            n_qubits,
            n_electrons=BH_ACTIVE_ELECTRONS,
            spin_ordering="interleaved",
            validation_tol=1.0e-8,
        )
        grouping_covariances, grouping_expectations = (
            covariance_dictionary_from_state(
                terms, grouping_state, n_qubits, "CISD"
            )
        )
        reporting_covariances, reporting_expectations = (
            covariance_dictionary_from_state(
                terms, exact_state, n_qubits, "exact"
            )
        )
        print(f"CISD determinant count: {cisd_metadata['basis_size']}")
    else:
        grouping_energy = exact_energy
        grouping_covariances, grouping_expectations = (
            covariance_dictionary_from_state(
                terms, exact_state, n_qubits, "exact"
            )
        )
        reporting_covariances = grouping_covariances
        reporting_expectations = grouping_expectations

    grouping_energy_check = varsi.hamiltonian_expectation(
        all_terms, grouping_expectations
    )
    reporting_energy_check = varsi.hamiltonian_expectation(
        all_terms, reporting_expectations
    )
    if not np.isclose(grouping_energy_check, grouping_energy, atol=1.0e-7):
        raise RuntimeError("Grouping-state covariance expectations are inconsistent.")
    if not np.isclose(reporting_energy_check, exact_energy, atol=1.0e-7):
        raise RuntimeError("Exact covariance expectations are inconsistent.")

    fast_context = varsi.build_fast_context(
        terms, grouping_covariances, CONDITION
    )
    binary_terms_by_key, pauli_terms_by_key, ics_covariances = (
        varsi.build_ics_bridge(
            tequila_hamiltonian, terms, grouping_covariances
        )
    )

    print(f"Grouping-state energy: {float(np.real(grouping_energy)):.12f}")
    print(f"Exact energy: {float(np.real(exact_energy)):.12f}")
    print(f"Number of qubits: {n_qubits}")
    print(f"Number of measurable Pauli words: {len(terms)}")
    return {
        "terms": terms,
        "grouping_covariances": grouping_covariances,
        "reporting_covariances": reporting_covariances,
        "grouping_method": method,
        "fast_context": fast_context,
        "binary_terms_by_key": binary_terms_by_key,
        "pauli_terms_by_key": pauli_terms_by_key,
        "ics_covariances": ics_covariances,
    }


def get_non_overlapping_groupings(problem):
    if problem["grouping_method"] == "CISD":
        print("Using CISD covariances for grouping and exact covariances for reporting results.")
    else:
        print("Using exact covariances for grouping and reporting results.")
    results = varsi.run_fast_varsi_methods(
        problem["fast_context"], FAST_OPTIONS
    )
    return {result.method: result for result in results}


def get_overlapping_ics_groupings(non_overlapping, problem):
    results, failures = varsi.run_ics_methods(
        list(non_overlapping.values()),
        problem["binary_terms_by_key"],
        problem["pauli_terms_by_key"],
        problem["grouping_covariances"],
        problem["ics_covariances"],
        CONDITION,
    )
    if failures:
        raise RuntimeError(f"ICS failed: {failures}")
    results_by_method = {result.method: result for result in results}
    ics_order = (
        "SI-ICS",
        "VarSI-G-ICS",
        "VarSI-O-ICS",
        "VarSI-R-ICS",
        "VarSI-OR-ICS",
    )
    ics_results = {
        name: results_by_method[name] for name in ics_order
    }
    allocations = {
        name: np.asarray(result.sample_ratios, dtype=float)
        for name, result in ics_results.items()
    }
    return ics_results, allocations


def summarize_groupings(groupings, covariances, allocations=None):
    rows = []
    for name, result in groupings.items():
        groups = result.groups
        variances = varsi.group_variances_from_covariances(
            groups, covariances
        )
        row = {
            "method": name,
            "groups": len(groups),
            "eps^2 M (exact)": varsi.eps_sq_m_from_variances(variances),
        }
        if allocations is not None:
            row["allocation sum"] = float(np.sum(allocations[name]))
        rows.append(row)
    return pd.DataFrame(rows).set_index("method")

## 1. Tequila

Build the molecular Hamiltonian.

In [ ]:
tequila_geometry = "\n".join(
    f"{symbol} {x:.8f} {y:.8f} {z:.8f}"
    for symbol, (x, y, z) in BH_GEOMETRY
)
tequila_molecule = tq.chemistry.Molecule(
    geometry=tequila_geometry,
    basis_set="sto-3g",
    transformation="JordanWigner",
    backend="pyscf",
    frozen_core=True,
)
tequila_hamiltonian = tequila_molecule.make_hamiltonian()
tequila_qubit_operator = tequila_hamiltonian.to_openfermion()
print(f"Number of qubits: {count_qubits(tequila_qubit_operator)}")
print(f"Number of Pauli words: {len(tequila_qubit_operator.terms)}")

Build the selected grouping covariances and the exact reporting covariances. Change the method below to `"CISD"` to use CISD covariances for grouping.

In [ ]:
tequila_covariance_method = "exact"  # default grouping covariances
# tequila_covariance_method = "CISD"  # uncomment to override
tequila_problem = build_covariance_problem(
    tequila_qubit_operator, method=tequila_covariance_method
)

Get the non-overlapping SI and VarSI groupings.

In [ ]:
tequila_non_overlapping = get_non_overlapping_groupings(tequila_problem)
summarize_groupings(
    tequila_non_overlapping, tequila_problem["reporting_covariances"]
)

Get the overlapping ICS groupings.

In [ ]:
tequila_ics, tequila_allocations = get_overlapping_ics_groupings(
    tequila_non_overlapping, tequila_problem
)
summarize_groupings(
    tequila_ics,
    tequila_problem["reporting_covariances"],
    tequila_allocations,
)

## 2. PennyLane

Build the molecular Hamiltonian.

In [ ]:
import pennylane as qml

pennylane_molecule = qml.qchem.Molecule(
    BH_SYMBOLS,
    BH_COORDINATES,
    charge=0,
    mult=1,
    basis_name="sto-3g",
    unit="angstrom",
)
pennylane_hamiltonian, pennylane_n_qubits = qml.qchem.molecular_hamiltonian(
    pennylane_molecule,
    method="dhf",
    active_electrons=BH_ACTIVE_ELECTRONS,
    active_orbitals=len(BH_ACTIVE_ORBITALS),
    mapping="jordan_wigner",
)
pennylane_qubit_operator = qml.to_openfermion(pennylane_hamiltonian)
print(f"Number of qubits: {pennylane_n_qubits}")
print(f"Number of Pauli words: {len(pennylane_qubit_operator.terms)}")

Build the selected grouping covariances and the exact reporting covariances. Change the method below to `"CISD"` to use CISD covariances for grouping.

In [ ]:
pennylane_covariance_method = "exact"  # default grouping covariances
# pennylane_covariance_method = "CISD"  # uncomment to override
pennylane_problem = build_covariance_problem(
    pennylane_qubit_operator, method=pennylane_covariance_method
)

Get the non-overlapping SI and VarSI groupings.

In [ ]:
pennylane_non_overlapping = get_non_overlapping_groupings(pennylane_problem)
summarize_groupings(
    pennylane_non_overlapping, pennylane_problem["reporting_covariances"]
)

Get the overlapping ICS groupings.

In [ ]:
pennylane_ics, pennylane_allocations = get_overlapping_ics_groupings(
    pennylane_non_overlapping, pennylane_problem
)
summarize_groupings(
    pennylane_ics,
    pennylane_problem["reporting_covariances"],
    pennylane_allocations,
)

## 3. OpenFermion

Build the molecular Hamiltonian.

In [ ]:
from openfermion import MolecularData, get_fermion_operator, jordan_wigner
from pyscf import ao2mo, gto, scf

pyscf_molecule = gto.M(
    atom=BH_GEOMETRY,
    basis="sto-3g",
    charge=0,
    spin=0,
    unit="Angstrom",
    verbose=0,
)
mean_field = scf.RHF(pyscf_molecule).run()
if not mean_field.converged:
    raise RuntimeError("PySCF did not converge.")

mo_coeff = mean_field.mo_coeff
n_orbitals = mo_coeff.shape[1]
one_body_integrals = mo_coeff.T @ mean_field.get_hcore() @ mo_coeff
two_body_integrals = ao2mo.restore(
    1, ao2mo.kernel(pyscf_molecule, mo_coeff), n_orbitals
).transpose(0, 2, 3, 1)

openfermion_molecule = MolecularData(
    BH_GEOMETRY,
    "sto-3g",
    multiplicity=1,
    charge=0,
    description="BH_R1p232_frozen_core",
)
openfermion_molecule.n_orbitals = n_orbitals
openfermion_molecule.n_qubits = 2 * n_orbitals
openfermion_molecule.n_electrons = pyscf_molecule.nelectron
openfermion_molecule.nuclear_repulsion = pyscf_molecule.energy_nuc()
openfermion_molecule.hf_energy = mean_field.e_tot
openfermion_molecule.one_body_integrals = one_body_integrals
openfermion_molecule.two_body_integrals = np.asarray(
    two_body_integrals, order="C"
)
active_space_hamiltonian = openfermion_molecule.get_molecular_hamiltonian(
    occupied_indices=BH_FROZEN_ORBITALS,
    active_indices=BH_ACTIVE_ORBITALS,
)
openfermion_fermion_operator = get_fermion_operator(
    active_space_hamiltonian
)
openfermion_qubit_operator = jordan_wigner(openfermion_fermion_operator)
print(f"Number of qubits: {count_qubits(openfermion_qubit_operator)}")
print(f"Number of Pauli words: {len(openfermion_qubit_operator.terms)}")

Build the selected grouping covariances and the exact reporting covariances. Change the method below to `"CISD"` to use CISD covariances for grouping.

In [ ]:
openfermion_covariance_method = "exact"  # default grouping covariances
# openfermion_covariance_method = "CISD"  # uncomment to override
openfermion_problem = build_covariance_problem(
    openfermion_qubit_operator, method=openfermion_covariance_method
)

Get the non-overlapping SI and VarSI groupings.

In [ ]:
openfermion_non_overlapping = get_non_overlapping_groupings(openfermion_problem)
summarize_groupings(
    openfermion_non_overlapping, openfermion_problem["reporting_covariances"]
)

Get the overlapping ICS groupings.

In [ ]:
openfermion_ics, openfermion_allocations = get_overlapping_ics_groupings(
    openfermion_non_overlapping, openfermion_problem
)
summarize_groupings(
    openfermion_ics,
    openfermion_problem["reporting_covariances"],
    openfermion_allocations,
)